In [25]:
import pandas as pd
from datetime import date
from sklearn.model_selection import GroupShuffleSplit
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, StandardScaler

# Data Retrieval

In [26]:
url = (
    "https://exoplanetarchive.ipac.caltech.edu/TAP/sync" # data source
    "?query=select+*+from+cumulative&format=csv" # query
)
df = pd.read_csv(url)
df.to_csv(f"data/raw/koi_cumulative_{date.today()}.csv", index=False) # file is dated 

In [27]:
print(df.shape)
print(df["koi_disposition"].value_counts())

(9564, 153)
koi_disposition
FALSE POSITIVE    4839
CONFIRMED         2748
CANDIDATE         1977
Name: count, dtype: int64


In [28]:
df.head()

,kepid,kepoi_name,kepler_name,ra,ra_err,ra_str,dec,dec_err,dec_str,koi_gmag,...,koi_fpflag_co,koi_fpflag_ec,koi_insol,koi_insol_err1,koi_insol_err2,koi_srho,koi_srho_err1,koi_srho_err2,koi_fittype,koi_score
0,10797460,K00752.01,Kepler-227 b,291.93423,0.0,19h27m44.22s,48.141651,0.0,+48d08m29.9s,15.890,...,0,0,93.59,29.45,-16.65,3.20796,0.33173,-1.09986,LS+MCMC,1.000
1,10797460,K00752.02,Kepler-227 c,291.93423,0.0,19h27m44.22s,48.141651,0.0,+48d08m29.9s,15.890,...,0,0,9.11,2.87,-1.62,3.02368,2.20489,-2.49638,LS+MCMC,0.969
2,10811496,K00753.01,NaN,297.00482,0.0,19h48m01.16s,48.134129,0.0,+48d08m02.9s,15.943,...,0,0,39.30,31.04,-10.49,7.29555,35.03293,-2.75453,LS+MCMC,0.000
3,10848459,K00754.01,NaN,285.53461,0.0,19h02m08.31s,48.285210,0.0,+48d17m06.8s,16.100,...,0,0,891.96,668.95,-230.35,0.22080,0.00917,-0.01837,LS+MCMC,0.000
4,10854555,K00755.01,Kepler-664 b,288.75488,0.0,19h15m01.17s,48.226200,0.0,+48d13m34.3s,16.015,...,0,0,926.16,874.33,-314.24,1.98635,2.71141,-1.74541,LS+MCMC,1.000


# Data Leakage Handling

In [29]:
known = df[df["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE"])].copy()
cands = df[df["koi_disposition"] == "CANDIDATE"].copy()

empty_cols = [c for c in df.columns if df[c].isna().all()]

id_and_label = ["kepid", "kepoi_name", "kepler_name", "koi_disposition"]

verdicts = [
    "koi_pdisposition", "koi_score",
    "koi_fpflag_nt", "koi_fpflag_ss", "koi_fpflag_co", "koi_fpflag_ec",
    "koi_comment", "koi_vet_stat", "koi_vet_date", "koi_disp_prov",
]

links = ["koi_datalink_dvr", "koi_datalink_dvs"]

not_useful = ["koi_time0bk", "koi_time0",
              "koi_time0bk_err1", "koi_time0bk_err2",
              "koi_time0_err1", "koi_time0_err2"]

uncertain = [
    "koi_bin_oedp_sig", "koi_count", "koi_tce_delivname", "koi_tce_plnt_num",
    "koi_fittype", "koi_parm_prov", "koi_limbdark_mod", "koi_trans_mod",
    "koi_sparprov", "koi_quarters", "ra", "dec",
]

drop_cols = empty_cols + id_and_label + verdicts + links + not_useful + uncertain

# only drop names that really exist in the table
drop_cols = [c for c in drop_cols if c in df.columns]

features = [c for c in df.columns if c not in drop_cols]
print(len(features))
print(features)

more_drops = [
    "ra_err", "ra_str", "dec_err", "dec_str", "koi_delivname",
    "koi_fwm_sra", "koi_fwm_sra_err", "koi_fwm_sdec", "koi_fwm_sdec_err",
]
features = [c for c in features if c not in more_drops]
print("Features left:", len(features))

95
['ra_err', 'ra_str', 'dec_err', 'dec_str', 'koi_gmag', 'koi_rmag', 'koi_imag', 'koi_zmag', 'koi_jmag', 'koi_jmag_err', 'koi_hmag', 'koi_hmag_err', 'koi_kmag', 'koi_kmag_err', 'koi_kepmag', 'koi_delivname', 'koi_num_transits', 'koi_max_sngle_ev', 'koi_max_mult_ev', 'koi_ldm_coeff4', 'koi_ldm_coeff3', 'koi_ldm_coeff2', 'koi_ldm_coeff1', 'koi_model_snr', 'koi_eccen', 'koi_prad', 'koi_prad_err1', 'koi_prad_err2', 'koi_sma', 'koi_impact', 'koi_impact_err1', 'koi_impact_err2', 'koi_duration', 'koi_duration_err1', 'koi_duration_err2', 'koi_depth', 'koi_depth_err1', 'koi_depth_err2', 'koi_period', 'koi_period_err1', 'koi_period_err2', 'koi_ror', 'koi_ror_err1', 'koi_ror_err2', 'koi_dor', 'koi_dor_err1', 'koi_dor_err2', 'koi_incl', 'koi_teq', 'koi_steff', 'koi_steff_err1', 'koi_steff_err2', 'koi_slogg', 'koi_slogg_err1', 'koi_slogg_err2', 'koi_smet', 'koi_smet_err1', 'koi_smet_err2', 'koi_srad', 'koi_srad_err1', 'koi_srad_err2', 'koi_smass', 'koi_smass_err1', 'koi_smass_err2', 'koi_fwm_stat_

# Preprocessing (Pre-Split)

In [30]:
# target column (1 = planet, 0 = false positive)
known["target"] = (known["koi_disposition"] == "CONFIRMED").astype(int)
print(known["target"].value_counts())

target
0    4839
1    2748
Name: count, dtype: int64


In [31]:
# remove duplicates
print("Duplicate names:", known["kepoi_name"].duplicated().sum())
print("Duplicate feature rows:", known.duplicated(subset=features).sum())

Duplicate names: 0
Duplicate feature rows: 0


In [32]:
counts = known["kepid"].value_counts()
print("Unique stars:", known["kepid"].nunique())
print("Stars with more than one object:", (counts > 1).sum())

Unique stars: 6641
Stars with more than one object: 651


In [33]:
# check for constant columns?
print(known[features].nunique().sort_values().head(10))
constant_cols = ["koi_eccen", "koi_ldm_coeff3", "koi_ldm_coeff4"]
features = [c for c in features if c not in constant_cols]
print(len(features))

koi_eccen              1
koi_ldm_coeff3         1
koi_ldm_coeff4         1
koi_smet_err1         41
koi_smet_err2         48
koi_jmag_err          97
koi_smet             116
koi_hmag_err         118
koi_kmag_err         141
koi_dikco_mra_err    159
dtype: int64
83


In [34]:
# check for out of range values
print(known[features].describe().T[["min", "max"]])

                         min           max
koi_gmag        7.225000e+00  2.115000e+01
koi_rmag        7.101000e+00  1.996000e+01
koi_imag        7.627000e+00  1.990000e+01
koi_zmag        7.657000e+00  1.740300e+01
koi_jmag        5.431000e+00  1.737200e+01
...                      ...           ...
koi_insol_err1  0.000000e+00  3.617133e+06
koi_insol_err2 -5.600031e+06  0.000000e+00
koi_srho        4.000000e-05  9.808542e+02
koi_srho_err1   0.000000e+00  8.352423e+02
koi_srho_err2  -6.960892e+02  0.000000e+00

[83 rows x 2 columns]


# Train-Test Split

In [35]:
# 60-20-20
X = known[features]
y = known["target"]
groups = known["kepid"]   # the star each row belongs to

# Split 1: cut off 20% as the test set (whole stars, not single rows)
splitter_1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
trainval_idx, test_idx = next(splitter_1.split(X, y, groups))

X_trainval = X.iloc[trainval_idx]
y_trainval = y.iloc[trainval_idx]
groups_trainval = groups.iloc[trainval_idx]

X_test = X.iloc[test_idx]
y_test = y.iloc[test_idx]

# Split 2: from the remaining 80%, cut off 25% as validation
# (25% of 80% = 20% of the total)
splitter_2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(splitter_2.split(X_trainval, y_trainval, groups_trainval))

X_train = X_trainval.iloc[train_idx]
y_train = y_trainval.iloc[train_idx]

X_val = X_trainval.iloc[val_idx]
y_val = y_trainval.iloc[val_idx]

stars_train = groups_trainval.iloc[train_idx]
stars_val = groups_trainval.iloc[val_idx]
stars_test = groups.iloc[test_idx]

In [36]:
# check the split
# 1. Sizes
print("train:", len(X_train), "val:", len(X_val), "test:", len(X_test))

# 2. Share of planets in each set (overall it is about 36%)
print("train planets:", round(y_train.mean(), 3))
print("val planets:  ", round(y_val.mean(), 3))
print("test planets: ", round(y_test.mean(), 3))

# 3. No star appears in two sets (all three numbers should be 0)
print(len(set(stars_train) & set(stars_val)))
print(len(set(stars_train) & set(stars_test)))
print(len(set(stars_val) & set(stars_test)))

train: 4519 val: 1522 test: 1546
train planets: 0.352
val planets:   0.376
test planets:  0.38
0
0
0


# Preprocessing (Post-split)

In [37]:
# check data gaps
missing = (X_train.isna().mean() * 100).round(1)
print(missing[missing > 0].sort_values(ascending=False))
print("Columns fully empty in train:", (missing == 100).sum())

koi_fwm_stat_sig     12.7
koi_fwm_pdeco_err     9.7
koi_fwm_pdeco         9.7
koi_fwm_prao          9.6
koi_fwm_prao_err      9.6
                     ... 
koi_gmag              0.5
koi_kmag              0.4
koi_hmag              0.4
koi_jmag              0.4
koi_rmag              0.1
Length: 80, dtype: float64
Columns fully empty in train: 0


In [38]:
# check skew
skew = X_train.skew().abs().sort_values(ascending=False)
print(skew.head(15))

koi_depth_err2       64.180185
koi_depth_err1       64.180185
koi_prad_err2        61.285429
koi_fwm_prao_err     60.762907
koi_fwm_pdeco_err    59.934291
koi_prad             54.807218
koi_fwm_pdeco        48.616051
koi_insol_err2       45.793817
koi_insol            42.968121
koi_insol_err1       38.931434
koi_prad_err1        33.608722
koi_srad_err2        32.491249
koi_dicco_mra_err    29.338460
koi_dikco_mra_err    29.257836
koi_ror              25.540381
dtype: float64


In [39]:
def signed_log(x):
    return np.sign(x) * np.log1p(np.abs(x))

prep = Pipeline([
    ("fill_gaps", SimpleImputer(strategy="median")),
    ("log", FunctionTransformer(signed_log)),
    ("scale", StandardScaler()),
])

X_train_prep = prep.fit_transform(X_train)   # learns from train, then applies
X_val_prep = prep.transform(X_val)           # only applies, never learns
X_test_prep = prep.transform(X_test)         # same, but we won't look at it

In [40]:
print("NaNs left in train:", np.isnan(X_train_prep).sum())
print("NaNs left in val:  ", np.isnan(X_val_prep).sum())
print("Train mean (should be about 0):", round(X_train_prep.mean(), 3))
print("Train std  (should be about 1):", round(X_train_prep.std(), 3))
print("Val mean:", round(X_val_prep.mean(), 3))
print("Val std: ", round(X_val_prep.std(), 3))
print("Columns before and after:", X_train.shape[1], X_train_prep.shape[1])

NaNs left in train: 0
NaNs left in val:   0
Train mean (should be about 0): 0.0
Train std  (should be about 1): 1.0
Val mean: -0.013
Val std:  0.997
Columns before and after: 83 83


# Model Fit

In [42]:
# baseline (logistic regression)

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_prep, y_train)

val_scores = model.predict_proba(X_val_prep)[:, 1]

from sklearn.metrics import accuracy_score

import numpy as np

def precision_at_k(y_true, scores, k):
    order = np.argsort(scores)[::-1]   # highest score first
    top_k = order[:k]                  # positions of the k best-ranked objects
    return y_true.iloc[top_k].mean()   # share of real planets among them

print("Accuracy:", round(accuracy_score(y_val, val_scores > 0.5), 3))

for k in [50, 100, 300]:
    print("Precision at", k, ":", round(precision_at_k(y_val, val_scores, k), 3))

Accuracy: 0.957
Precision at 50 : 1.0
Precision at 100 : 1.0
Precision at 300 : 1.0


In [43]:
# 1. Does it stay good deeper into the list?
for k in [400, 500, 600, 700]:
    print("Precision at", k, ":", round(precision_at_k(y_val, val_scores, k), 3))

# 2. Which columns does the model rely on most?
import pandas as pd
coefs = pd.Series(model.coef_[0], index=X_train.columns)
print(coefs.abs().sort_values(ascending=False).head(10))

Precision at 400 : 0.995
Precision at 500 : 0.984
Precision at 600 : 0.913
Precision at 700 : 0.804
koi_insol           3.362521
koi_max_sngle_ev    2.932948
koi_dikco_msky      2.023711
koi_impact          1.848324
koi_ror             1.790774
koi_max_mult_ev     1.632487
koi_duration        1.600774
koi_impact_err2     1.487373
koi_num_transits    1.414979
koi_teq             1.343318
dtype: float64


In [45]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    eval_metric="logloss",
    random_state=42,
)
xgb.fit(X_train, y_train)

xgb_scores = xgb.predict_proba(X_val)[:, 1]

In [46]:
from sklearn.metrics import log_loss

np.random.seed(42)
all_scores = {
    "Random": np.random.rand(len(y_val)),
    "SNR only": X_val["koi_model_snr"].fillna(0).values,
    "Logistic regression": val_scores,
    "XGBoost": xgb_scores,
}

ks = [50, 100, 300, 500, 572, 650]

print("Precision at k")
for name, s in all_scores.items():
    row = [round(precision_at_k(y_val, s, k), 3) for k in ks]
    print(name, row)

print("\nLog loss (lower is better)")
for name in ["Logistic regression", "XGBoost"]:
    print(name, round(log_loss(y_val, all_scores[name]), 4))

Precision at k
Random [0.34, 0.36, 0.34, 0.342, 0.348, 0.355]
SNR only [0.04, 0.08, 0.147, 0.274, 0.313, 0.34]
Logistic regression [1.0, 1.0, 1.0, 0.984, 0.939, 0.855]
XGBoost [1.0, 1.0, 1.0, 0.996, 0.958, 0.863]

Log loss (lower is better)
Logistic regression 0.1306
XGBoost 0.0919


In [47]:
print(log_loss(y_train, xgb.predict_proba(X_train)[:, 1]))

0.010317742349203136


In [48]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

known_dev = pd.concat([X_train, X_val])
stars_dev = pd.concat([stars_train, stars_val])

adv_X = pd.concat([known_dev, cands[features]]).reset_index(drop=True)
adv_y = np.array([0] * len(known_dev) + [1] * len(cands))   # 0 = known, 1 = candidate
adv_groups = pd.concat([stars_dev, cands["kepid"]]).reset_index(drop=True)

adv_model = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                          eval_metric="logloss", random_state=42)

adv_scores = cross_val_predict(adv_model, adv_X, adv_y, groups=adv_groups,
                               cv=GroupKFold(n_splits=5), method="predict_proba")[:, 1]

print("AUC:", round(roc_auc_score(adv_y, adv_scores), 3))

# Which columns separate the two groups?
adv_model.fit(adv_X, adv_y)
importance = pd.Series(adv_model.feature_importances_, index=features)
print(importance.sort_values(ascending=False).head(10))

AUC: 0.904
koi_model_snr         0.148468
koi_max_mult_ev       0.074038
koi_dikco_msky        0.048786
koi_dor               0.032822
koi_dicco_msky_err    0.022538
koi_dor_err2          0.021666
koi_dor_err1          0.020850
koi_sma               0.020528
koi_prad              0.019285
koi_dikco_msky_err    0.018263
dtype: float32


In [49]:
low_cut = X_train["koi_model_snr"].quantile(0.25)   # cut-off from training data only

print("Median SNR, known:     ", round(known_dev["koi_model_snr"].median(), 1))
print("Median SNR, candidates:", round(cands["koi_model_snr"].median(), 1))
print("Share of known below cut-off:     ", round((known_dev["koi_model_snr"] < low_cut).mean(), 2))
print("Share of candidates below cut-off:", round((cands["koi_model_snr"] < low_cut).mean(), 2))

Median SNR, known:      31.1
Median SNR, candidates: 11.0
Share of known below cut-off:      0.24
Share of candidates below cut-off: 0.72


In [50]:
from sklearn.metrics import roc_auc_score

weak = X_val["koi_model_snr"] < low_cut
strong = ~weak

for name, mask in [("weak signal", weak), ("strong signal", strong)]:
    print(name, "| objects:", mask.sum(), "| planets:", y_val[mask].sum(),
          "| AUC:", round(roc_auc_score(y_val[mask], xgb_scores[mask]), 3))

weak signal | objects: 366 | planets: 83 | AUC: 0.989
strong signal | objects: 1156 | planets: 489 | AUC: 0.995


In [51]:
table = pd.DataFrame({"predicted": xgb_scores, "actual": y_val.values})
table["group"] = pd.cut(table["predicted"], [0, 0.1, 0.5, 0.9, 1.0], include_lowest=True)
print(table.groupby("group", observed=True).agg(
    count=("actual", "size"),
    avg_predicted=("predicted", "mean"),
    share_real_planets=("actual", "mean")).round(3))

               count  avg_predicted  share_real_planets
group                                                  
(-0.001, 0.1]    911          0.005               0.013
(0.1, 0.5]        41          0.261               0.317
(0.5, 0.9]        37          0.740               0.568
(0.9, 1.0]       533          0.991               0.987


In [52]:
xgb_small = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    eval_metric="logloss",
    random_state=42,
)
xgb_small.fit(X_train, y_train)

small_scores = xgb_small.predict_proba(X_val)[:, 1]
print("Validation log loss:", round(log_loss(y_val, small_scores), 4))
print("Training log loss:  ", round(log_loss(y_train, xgb_small.predict_proba(X_train)[:, 1]), 4))
print("Precision at 572:   ", round(precision_at_k(y_val, small_scores, 572), 3))

Validation log loss: 0.09
Training log loss:   0.0253
Precision at 572:    0.958


In [53]:
def test_extra_column(col):
    cols = features + [col]
    model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3,
                          eval_metric="logloss", random_state=42)
    model.fit(known.loc[X_train.index, cols], y_train)
    scores = model.predict_proba(known.loc[X_val.index, cols])[:, 1]
    print(col)
    print("  Validation log loss:", round(log_loss(y_val, scores), 4))
    print("  Precision at 572:   ", round(precision_at_k(y_val, scores, 572), 3))

test_extra_column("koi_bin_oedp_sig")

koi_bin_oedp_sig
  Validation log loss: 0.0904
  Precision at 572:    0.96


In [54]:
for col in ["koi_count", "koi_tce_plnt_num", "ra", "dec"]:
    test_extra_column(col)

koi_count
  Validation log loss: 0.0912
  Precision at 572:    0.958
koi_tce_plnt_num
  Validation log loss: 0.089
  Precision at 572:    0.962
ra
  Validation log loss: 0.0896
  Precision at 572:    0.958
dec
  Validation log loss: 0.0898
  Precision at 572:    0.958


# Final Model

In [55]:
final_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    eval_metric="logloss",
    random_state=42,
)
final_model.fit(X_trainval, y_trainval)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [56]:
test_scores = final_model.predict_proba(X_test)[:, 1]
n_planets = int(y_test.sum())
print("Objects in test:", len(y_test), "| planets:", n_planets)

for k in [50, 100, 300, n_planets]:
    print("Precision at", k, ":", round(precision_at_k(y_test, test_scores, k), 3))

print("Log loss:", round(log_loss(y_test, test_scores), 4))
print("Accuracy:", round(accuracy_score(y_test, test_scores > 0.5), 3))

Objects in test: 1546 | planets: 587
Precision at 50 : 1.0
Precision at 100 : 1.0
Precision at 300 : 0.997
Precision at 587 : 0.973
Log loss: 0.0729
Accuracy: 0.979


# Error analysis

In [57]:
results = known.loc[X_test.index, ["kepoi_name", "koi_comment", "koi_model_snr",
                                   "koi_dikco_msky", "koi_impact", "koi_period", "koi_prad"]].copy()
results["score"] = test_scores
results["actual"] = y_test.values

# False positives the model was most sure were planets
fooled = results[results["actual"] == 0].sort_values("score", ascending=False).head(15)
print(fooled.round(2).to_string())

# Real planets the model gave the lowest scores
missed = results[results["actual"] == 1].sort_values("score").head(15)
print(missed.round(2).to_string())

     kepoi_name                                      koi_comment  koi_model_snr  koi_dikco_msky  koi_impact  koi_period  koi_prad  score  actual
1869  K02307.01                                       NO_COMMENT           24.2            0.24        0.97        4.15      3.03   1.00       0
2742  K00379.01                           CENT_UNRESOLVED_OFFSET           49.1            0.89        0.16        6.72      2.78   0.97       0
2197  K02601.01                           CENT_UNRESOLVED_OFFSET           42.9            0.96        0.66        2.62      1.64   0.92       0
7437  K02768.03                                       NO_COMMENT           12.7            0.31        0.55       11.83      1.17   0.92       0
4188  K03194.01                                   CENT_SATURATED           16.6            0.48        0.80       10.81      1.46   0.91       0
7419  K06137.02                        CENT_KIC_POS---HALO_GHOST           10.6            0.33        0.07        7.38      1.22 

# Save Model

In [58]:
import joblib, os
os.makedirs("models", exist_ok=True)
joblib.dump(final_model, "models/xgb_final.joblib")
joblib.dump(features, "models/feature_list.joblib")

['models/feature_list.joblib']

# Build app

In [61]:
cand_scores = final_model.predict_proba(cands[features])[:, 1]

ranking = cands[["kepoi_name", "kepid", "koi_model_snr", "koi_period", "koi_prad"]].copy()
ranking["planet_probability"] = cand_scores
ranking = ranking.sort_values("planet_probability", ascending=False).reset_index(drop=True)
ranking["rank"] = ranking.index + 1

ranking.to_csv("data/processed/candidate_ranking.csv", index=False)
print(ranking.head(10).round(3))

  kepoi_name     kepid  koi_model_snr  koi_period  koi_prad  \
0  K01972.01  11253711           39.8      17.791      2.88   
1  K01656.01   4471747           15.4      51.390      3.62   
2  K01590.01   5542466           23.6      12.890      1.93   
3  K00266.02   7375348           15.0      47.744      1.74   
4  K02657.01   9283156           15.1       5.224      0.58   
5  K00298.02  12785320           22.9      57.384      1.62   
6  K03448.01   3348216           12.6      32.511      2.62   
7  K03323.01   5193659           12.5      14.320      2.06   
8  K04235.01  10794974           14.8      66.643      2.99   
9  K02124.01  11462341           18.1      42.338      1.45   

   planet_probability  rank  
0               0.997     1  
1               0.996     2  
2               0.996     3  
3               0.996     4  
4               0.996     5  
5               0.996     6  
6               0.996     7  
7               0.995     8  
8               0.995     9  
9     

In [62]:
# 1. How are the scores spread, compared with the test set?
bands = [0, 0.1, 0.5, 0.9, 1.0]
print("Candidates:")
print(pd.cut(ranking["planet_probability"], bands, include_lowest=True).value_counts(normalize=True).sort_index().round(3))
print("Test set:")
print(pd.cut(pd.Series(test_scores), bands, include_lowest=True).value_counts(normalize=True).sort_index().round(3))

# 2. Does our ranking agree with the Kepler team's own score?
from scipy.stats import spearmanr
both = cands.assign(ours=cand_scores).dropna(subset=["koi_score"])
print("Rank agreement with koi_score:", round(spearmanr(both["ours"], both["koi_score"])[0], 3))

# 3. Do the top-ranked candidates look different from the bottom ones?
print(ranking.head(100)["koi_model_snr"].median(), ranking.tail(100)["koi_model_snr"].median())

Candidates:
planet_probability
(-0.001, 0.1]    0.467
(0.1, 0.5]       0.182
(0.5, 0.9]       0.181
(0.9, 1.0]       0.170
Name: proportion, dtype: float64
Test set:
(-0.001, 0.1]    0.587
(0.1, 0.5]       0.033
(0.5, 0.9]       0.036
(0.9, 1.0]       0.344
Name: proportion, dtype: float64
Rank agreement with koi_score: 0.417
17.35 8.2


In [64]:
import streamlit, pandas
print("streamlit==" + streamlit.__version__)
print("pandas==" + pandas.__version__)

streamlit==1.56.0
pandas==2.3.0
